In [599]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [600]:
df = pd.read_csv("dirty_cafe_sales.csv")
df.head()

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,ERROR,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,UNKNOWN,UNKNOWN,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11


In [601]:
df.shape

(10000, 8)

In [602]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   Transaction ID    10000 non-null  object
 1   Item              9667 non-null   object
 2   Quantity          9862 non-null   object
 3   Price Per Unit    9821 non-null   object
 4   Total Spent       9827 non-null   object
 5   Payment Method    7421 non-null   object
 6   Location          6735 non-null   object
 7   Transaction Date  9841 non-null   object
dtypes: object(8)
memory usage: 625.1+ KB


In [603]:
df.describe().T

,count,unique,top,freq
Transaction ID,10000,10000,TXN_1961373,1
Item,9667,10,Juice,1171
Quantity,9862,7,5,2013
Price Per Unit,9821,8,3.0,2429
Total Spent,9827,19,6.0,979
Payment Method,7421,5,Digital Wallet,2291
Location,6735,4,Takeaway,3022
Transaction Date,9841,367,UNKNOWN,159


In [604]:
df.isna().sum()

Transaction ID         0
Item                 333
Quantity             138
Price Per Unit       179
Total Spent          173
Payment Method      2579
Location            3265
Transaction Date     159
dtype: int64

### `Numeric` olan satırlarda işlem yapabilmek için `'ERROR', 'UNKNOWN', np.nan` değerlerini `np.nan` yapıp `pd.to_numeric` ile `float64` değerine çevirelim

### `for c in numeric_cols: df[c] = pd.to_numeric(df[c], errors="coerce")`
### kodu eğer satırlardaki değerler `sayı` ise `olduğu gibi` kabul eder
### satırlardaki değerler `sayı gibi görünüyorsa` "ör : '6'(string)" `numeric değere` çevirir
### satırlardaki değerler `sayı dışında` herhangi bir şey ise `np.nan` olarak çevirir

In [605]:
# object tipinde olan numeric verileri numeric tipe çevirelim
numeric_cols = ["Price Per Unit", "Total Spent", "Quantity"]
for c in numeric_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

In [606]:
df

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2.0,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4.0,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4.0,1.0,NaN,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2.0,5.0,10.0,UNKNOWN,UNKNOWN,2023-04-27
4,TXN_3160411,Coffee,2.0,2.0,4.0,Digital Wallet,In-store,2023-06-11
...,...,...,...,...,...,...,...,...
9995,TXN_7672686,Coffee,2.0,2.0,4.0,NaN,UNKNOWN,2023-08-30
9996,TXN_9659401,NaN,3.0,NaN,3.0,Digital Wallet,NaN,2023-06-02
9997,TXN_5255387,Coffee,4.0,2.0,8.0,Digital Wallet,NaN,2023-03-02
9998,TXN_7695629,Cookie,3.0,NaN,3.0,Digital Wallet,NaN,2023-12-02


In [607]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Transaction ID    10000 non-null  object 
 1   Item              9667 non-null   object 
 2   Quantity          9521 non-null   float64
 3   Price Per Unit    9467 non-null   float64
 4   Total Spent       9498 non-null   float64
 5   Payment Method    7421 non-null   object 
 6   Location          6735 non-null   object 
 7   Transaction Date  9841 non-null   object 
dtypes: float64(3), object(5)
memory usage: 625.1+ KB


### `Price Per Unit`sütunundaki bazı `NaN` değerindeki satırları `Total Spent` sütununu `Quantity` sütununa bölerek dolduralım

In [608]:
df["Price Per Unit"].isna().sum()

np.int64(533)

In [609]:
# Price Per Unit sütunundaki eksik değerleri Total Spent ve Quantity sütunlarını kullanarak dolduralım
# maskeleme yaparak eksik Price Per Unit değerlerini hesaplayalım
mask = df["Price Per Unit"].isna() & df["Quantity"].notna() & df["Total Spent"].notna()
# eksik Price Per Unit değerlerini hesapla ve ata
df.loc[mask, "Price Per Unit"] = df.loc[mask, "Total Spent"] / df.loc[mask, "Quantity"]

In [610]:
df["Price Per Unit"].isna().sum()

np.int64(38)

### `Price Per Unit`sütunundaki bazı `NaN` değerindeki satırları `Item` sütunundaki karşılığa göre dolduralım

In [611]:
# Price Per Unit sütunundaki eksik değerleri Item sütunundaki bilgilere göre dolduralım
item_price_map = {
    "Cookie" : 1.0,
    "Coffee" : 2.0,
    "Tea" : 1.5,
    "Juice" : 3.0,
    "Cake" : 3.0,
    "Smoothie" : 4.0,
    "Sandwich" : 4.0,
    "Salad" : 5.0
}

# maskeleme yaparak eksik Price Per Unit değerlerini hesaplayalım
mask = df["Price Per Unit"].isna() & df["Item"].isin(item_price_map.keys())
# elsik Price Per Unit değerlerini ismine göre ata
df.loc[mask, "Price Per Unit"] = df.loc[mask, "Item"].map(item_price_map)

In [612]:
df["Price Per Unit"].isna().sum()

np.int64(6)

### `Quantity`sütunundaki bazı `NaN` değerindeki satırları `Total Spent` sütununu `Price Per Unit` sütununa bölerek dolduralım

In [613]:
df["Quantity"].isna().sum()

np.int64(479)

In [614]:
# maskeleme yaparak eksik Quantity değerlerini hesaplayalım
mask = df["Quantity"].isna() & df["Price Per Unit"].notna() & df["Total Spent"].notna()
# eksik Quantity değerlerini hesapla ve ata
df.loc[mask, "Quantity"] = df.loc[mask, "Total Spent"] / df.loc[mask, "Price Per Unit"]

In [615]:
df["Quantity"].isna().sum()

np.int64(23)

### `Total Spent` sütunundaki bazı `NaN` değerindeki satırları `Quantity` sütunu ile `Price Per Unit` satırlarını çarparak dolduralım

In [616]:
df["Total Spent"].isna().sum()

np.int64(502)

In [617]:
# maskeleme yaparak eksik Total Spent değerlerini hesaplayalım
mask = df["Total Spent"].isna() & df["Price Per Unit"].notna() & df["Quantity"].notna()
# eksik Total Spent değerlerini hesapla ve ata
df.loc[mask, "Total Spent"] = df.loc[mask, "Price Per Unit"] * df.loc[mask, "Quantity"]

In [618]:
df["Total Spent"].isna().sum()

np.int64(23)

In [619]:
df.isna().sum()

Transaction ID         0
Item                 333
Quantity              23
Price Per Unit         6
Total Spent           23
Payment Method      2579
Location            3265
Transaction Date     159
dtype: int64

### `Item` sütunundaki `'ERROR', 'UNKNOWN', np.nan` olan satırları `Price Per Unit` sütununa göre dolduralım

In [620]:
df[(df["Item"] == "UNKNOWN") | (df["Item"] == "ERROR") | (df["Item"].isna())]

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
6,TXN_4433211,UNKNOWN,3.0,3.0,9.0,ERROR,Takeaway,2023-10-06
8,TXN_4717867,NaN,5.0,3.0,15.0,NaN,Takeaway,2023-07-28
14,TXN_8915701,ERROR,2.0,1.5,3.0,NaN,In-store,2023-03-21
30,TXN_1736287,NaN,5.0,2.0,10.0,Digital Wallet,NaN,2023-06-02
31,TXN_8927252,UNKNOWN,2.0,1.0,2.0,Credit Card,ERROR,2023-11-06
...,...,...,...,...,...,...,...,...
9951,TXN_4122925,ERROR,4.0,1.0,4.0,NaN,Takeaway,2023-10-20
9958,TXN_4125474,ERROR,2.0,5.0,10.0,Credit Card,In-store,2023-08-02
9981,TXN_4583012,ERROR,5.0,4.0,20.0,Digital Wallet,NaN,2023-02-27
9994,TXN_7851634,UNKNOWN,4.0,4.0,16.0,NaN,NaN,2023-01-08


In [621]:
# Price Per Unit sütunundaki problemli değerleri NaN ile değiştirelim
price_to_item = {
    2.0 : "Coffee",
    3.0 : "Juice",
    1.0 : "Cookie",
    5.0 : "Salad",
    4.0 : "Sandwich",
    1.5 : "Tea"
}

problem_items = ["ERROR", "UNKNOWN", np.nan]

for price, item in price_to_item.items():
    df.loc[(df["Price Per Unit"] == price) & (df["Item"].isin(problem_items)), "Item"] = item

In [622]:
df[(df["Item"] == "UNKNOWN") | (df["Item"] == "ERROR") | (df["Item"].isna())]

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
1761,TXN_3611851,NaN,4.0,NaN,NaN,Credit Card,NaN,2023-02-09
2289,TXN_7524977,UNKNOWN,4.0,NaN,NaN,ERROR,NaN,2023-12-09
3779,TXN_7376255,UNKNOWN,NaN,NaN,25.0,NaN,In-store,2023-05-27
4152,TXN_9646000,ERROR,2.0,NaN,NaN,NaN,In-store,2023-12-14
7597,TXN_1082717,ERROR,NaN,NaN,9.0,Digital Wallet,In-store,2023-12-13
9819,TXN_1208561,NaN,NaN,NaN,20.0,Credit Card,NaN,2023-08-19


In [623]:
df.isna().sum()

Transaction ID         0
Item                   2
Quantity              23
Price Per Unit         6
Total Spent           23
Payment Method      2579
Location            3265
Transaction Date     159
dtype: int64

## Doldurulmayacak bazı satıları veri setimizden kalıcı olarak silelim

In [624]:
# subset ile sadece Price Per Unit sütunundaki NaN değerine sahip olan satırları düşürelim
df.dropna(subset=["Price Per Unit"], inplace=True)

In [625]:
# subset ile sadece Quantity sütunundaki NaN değerine sahip olan satırları düşürelim
df.dropna(subset = ["Quantity"], inplace=True)

In [626]:
df.isna().sum()

Transaction ID         0
Item                   0
Quantity               0
Price Per Unit         0
Total Spent            0
Payment Method      2570
Location            3257
Transaction Date     159
dtype: int64

In [627]:
cols = ["Payment Method", "Location"]
problem_values = ["UNKNOWN", "ERROR"]
# Belirtilen sütunlardaki problemli değerleri NaN ile değiştirelim
for c in cols:
    df[c] = df[c].replace(problem_values, np.nan)


In [628]:
df["Location"].value_counts()

Location
Takeaway    3016
In-store    3006
Name: count, dtype: int64

In [629]:
df["Payment Method"].value_counts()

Payment Method
Digital Wallet    2284
Credit Card       2268
Cash              2254
Name: count, dtype: int64

In [630]:
# belirtilen sütunlardaki NaN değerleri ilgili değerlerle dolduralım
df["Payment Method"].fillna("Digital Wallet", inplace=True)
df["Location"].fillna("Takeaway", inplace=True)

In [631]:
# Transaction Date sütununu düşürelim
df.drop(["Transaction Date"], axis=1, inplace=True)

In [632]:
df.isna().sum()

Transaction ID    0
Item              0
Quantity          0
Price Per Unit    0
Total Spent       0
Payment Method    0
Location          0
dtype: int64

In [633]:
df

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location
0,TXN_1961373,Coffee,2.0,2.0,4.0,Credit Card,Takeaway
1,TXN_4977031,Cake,4.0,3.0,12.0,Cash,In-store
2,TXN_4271903,Cookie,4.0,1.0,4.0,Credit Card,In-store
3,TXN_7034554,Salad,2.0,5.0,10.0,Digital Wallet,Takeaway
4,TXN_3160411,Coffee,2.0,2.0,4.0,Digital Wallet,In-store
...,...,...,...,...,...,...,...
9995,TXN_7672686,Coffee,2.0,2.0,4.0,Digital Wallet,Takeaway
9996,TXN_9659401,Cookie,3.0,1.0,3.0,Digital Wallet,Takeaway
9997,TXN_5255387,Coffee,4.0,2.0,8.0,Digital Wallet,Takeaway
9998,TXN_7695629,Cookie,3.0,1.0,3.0,Digital Wallet,Takeaway


## Temizlediğimiz .csv dosyasını farklı olarak kaydedelim

In [634]:
# ham dosyayı korumak için temizlenmiş veriyi yeni bir değişkene atayalım

#df_clean = df.copy()

In [635]:
# temizlenmiş veriyi csv dosyasına kaydedelim

#df_clean.to_csv("cleaned_cafe_sales.csv", index=False)